# Whisper(faster-whisper) 구조 뜯어보기: 소리가 글이 되기까지 — 입력 Log-Mel · 인코더 · 디코더 · 토큰

*2026-11-09, 이슈 #79. Demucs 구조 노트북(`demucs_anatomy_claude_code.ipynb`)과 같은 방식의 탐구입니다. 11-06 면담(원리 이해가 가장 중요, 시행착오를 있는 그대로)에 따라, 우리가 **실제로 겪은 현상**(환각·반복 루프·온도 폴백·VAD)을 구조로 설명하는 것이 목표입니다.*

**이 노트북의 답할 질문**
1. Whisper는 어떤 구조이고, 모델 크기(tiny → medium)가 커지면 **무엇이 얼마나** 커지나? (층 수·차원·파라미터)
2. 소리는 모델에 **어떤 모양**(Log-Mel 80×3000)으로 들어가고, 수업 4강의 STFT·Mel과 무엇이 같고 다른가?
3. 인코더는 무엇을 내놓고, 왜 **짧은 소리도 30초짜리와 같은 비용**일까?
4. 디코더는 글을 **어떻게 한 토큰씩** 만들고, 그 과정에서 확률(신뢰)이 어떻게 나오나?
5. 우리 프로젝트에서 겪은 현상(무음에서 말 만들기, 반복 루프, 온도 폴백, 노래에서 VAD가 가사를 지움)은 **구조의 어디에서 나오나?**

**표시**: **[확인]** 이 노트북을 실행해 확인한 값, **[일반 지식]** 논문·문서에서 알려진 내용, **[추정]** 해석, **[미확인]** 확인하지 못한 것.
**한계**: 입력은 **내 직접 쓴 예문의 TTS 소리와 합성 잡음**입니다(상업곡·내 목소리 없음). 결과를 실제 노래·다른 화자로 일반화하지 않습니다. 모델 파일은 CTranslate2로 변환된 `faster-whisper` 판이라 PyTorch 층 객체를 직접 열 수는 없어서, **모델 파일 안의 변수 이름과 모양을 직접 읽는 방법**으로 구조를 확인합니다.

## 0. 분석 대상과 원칙

| 대상 | 분석 목적 |
|---|---|
| [openai/whisper](https://github.com/openai/whisper) 논문·구조 | **모델 자체**: 인코더–디코더 트랜스포머, 입력 Log-Mel, 토큰 |
| [SYSTRAN/faster-whisper](https://github.com/SYSTRAN/faster-whisper) + [CTranslate2](https://github.com/OpenNMT/CTranslate2) | **우리가 실제로 쓰는 구현**: 같은 모델을 변환해 CPU에서 빠르게 돌리는 런타임 |
| [greenkey20/song-lingo](https://github.com/greenkey20/song-lingo)의 `stt.py` | **우리 서비스**가 모델을 어떻게 부르나(설정: `tiny`·`int8`·`beam_size=1`) |

**원칙**
- 수업 7강(`lesson07_whisper_minimal`)은 Whisper를 "불러 쓰는" 수준이었습니다. 이 노트북은 **그 안을 연다**는 점이 새롭습니다.
- 이 노트북에서는 **PyTorch·Demucs를 import하지 않습니다**(같은 커널에서 Whisper와 torch를 함께 불러 OpenMP 충돌로 커널이 죽은 적이 있음, 개발환경 이슈노트 #15). Demucs 쪽 탐구는 별도 노트북입니다.
- **실험은 반복합니다**(잡음 실험은 시드 3개). 한 번 나온 숫자를 일반화하지 않습니다.

## 1. 문제 정의와 큰 그림: 소리 → 글

**음성 인식(STT)** 은 소리 파형 $x(t)$를 받아 글자(토큰) 열 $y_1, y_2, \dots$ 를 내놓는 일입니다. Whisper는 이를 **인코더–디코더 트랜스포머** 하나로 풉니다.

```
16 kHz 파형 ──▶ 30초 창으로 자르기/채우기 ──▶ Log-Mel 스펙트로그램 (80 × 3000)
   (4강 STFT·Mel)            (고정 길이)             │
                                                      ▼  합성곱 2개 (시간 절반으로 줄임)
                                              인코더 N층 (셀프 어텐션 + FFN)
                                                      │  인코더 출력 (1500 × d)
                                                      ▼
   특수 토큰 <|startoftranscript|><|en|><|transcribe|> ──▶ 디코더 N층
      (셀프 어텐션 + 교차 어텐션 + FFN) ◀── 인코더 출력을 "교차 어텐션"으로 참조
                                                      │
                                                      ▼  다음 토큰의 확률 → 하나 고름 → 다시 입력에 붙임 (반복)
                                              " I want to explore the universe." <|endoftext|>
```

- **인코더**는 소리를 "의미 있는 표현"으로 바꾸고, **디코더**는 그 표현을 보면서 **글을 한 토큰씩** 씁니다(자기회귀).
- 디코더는 사실상 **언어 모델**입니다. 소리가 불분명해도 "그럴듯한 다음 말"을 고를 수 있어서, 이것이 환각과 "틀린 발음을 문맥으로 고쳐 적기"(우리 앱이 점수를 **발음 점수가 아니라 문장 인식 일치도**라고 하는 이유)의 뿌리입니다 **[추정]**.

## 2. 먼저 가설: 수업 개념은 Whisper의 어디에 있을까? (아래 실험으로 하나씩 확인)

| 수업 | 개념 | Whisper에서 찾을 곳 (가설) |
|---|---|---|
| 02·03·04강 | FFT, STFT, Mel, 스펙트로그램 | 모델 **입력** Log-Mel(`n_fft=400`, `hop=160`, 80 Mel) → 5절 |
| 006 CNN | 합성곱으로 국소 패턴 추출 | 인코더 맨 앞 `conv1`, `conv2`(시간 절반 줄이기) → 4·6절 |
| 010 Seq2Seq | 인코더로 압축 → 디코더로 생성 | `encoder` / `decoder` 두 덩어리 → 4절 |
| 011 Attention | $\mathrm{softmax}(QK^\top/\sqrt d)V$ | 인코더 **셀프** 어텐션, 디코더 셀프 + **교차** 어텐션 → 4절 |
| 012 Transformer | 어텐션 + 피드포워드 층을 쌓기 | 층 수(`tiny` 4 → `medium` 24)와 차원 → 4절 |
| 009 RNN/LSTM | 순서대로 상태 전달 | Whisper에는 **없을 것**(Transformer) → 4절에서 변수 이름으로 확인 |

그리고 우리 프로젝트와의 연결 가설입니다.
- **짧은 소리도 30초 창으로 채워 처리**하므로 인코더 시간은 소리 길이와 상관없이 같을 것 → 6절
- **무음에서도 디코더는 말을 만들어 낼** 수 있고, 그때 확률 지표(`avg_logprob`, `no_speech_prob`)가 나빠질 것 → 8절

In [ ]:
# ── 환경 확인 ──────────────────────────────────────────────────────────────
# 하는 일: 프로젝트 루트를 찾아 작업 폴더로 하고(노트북이 experiments/ 아래에 있어도 루트의 모듈을 import하려고),
#          faster-whisper·CTranslate2 버전, 쓸 모델 파일 위치, 예시 소리(내 직접 쓴 예문의 TTS)를 찾는다.
# [수업과 같음] faster-whisper 사용(수업 7강 lesson07_whisper_minimal)  [수업에 없음] CTranslate2 직접 호출, 모델 파일 읽기
# 주의: 이 노트북은 torch·demucs를 import하지 않는다(OpenMP 충돌 방지, 개발환경 이슈노트 #15).
import os, sys, time, inspect, re, glob, math, collections
from pathlib import Path
here = Path.cwd().resolve()
for p in [here, *here.parents]:
    if (p / "practice.py").exists():
        os.chdir(p); sys.path.insert(0, str(p)); break
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display      # 표를 예쁘게 보여 주는 노트북 함수
import faster_whisper, ctranslate2
from faster_whisper import WhisperModel
from faster_whisper.audio import decode_audio, pad_or_trim
from faster_whisper.tokenizer import Tokenizer
import theme, stt
theme.apply_style()

def model_dir(size):
    """models/ 에 내려받아 둔 faster-whisper 모델 폴더(없으면 None)."""
    hits = glob.glob(f"models/models--Systran--faster-whisper-{size}/snapshots/*/")
    return Path(hits[0]) if hits else None

SIZES = [s for s in ("tiny", "base", "small", "medium") if model_dir(s) and (model_dir(s) / "model.bin").exists()]

def find_sample():
    """예시 소리: 직접 쓴 예문(sentences.csv s01)의 TTS 캐시. 없으면 캐시에 있는 아무 wav."""
    for name in ("s01.wav",):
        p = Path("data/tts_cache") / name
        if p.exists():
            return p
    hits = sorted(Path("data/tts_cache").glob("*.wav"))
    if hits:
        return hits[0]
    raise FileNotFoundError("data/tts_cache/ 에 TTS wav가 없습니다. 앱에서 표준 발음을 한 번 만들거나 sentences.csv의 문장으로 tts.py를 실행하세요.")

SAMPLE = find_sample()
print("프로젝트 루트:", Path.cwd().name, "| faster-whisper", faster_whisper.__version__, "| CTranslate2", ctranslate2.__version__)
print("내려받아 둔 모델:", SIZES, "| 예시 소리:", SAMPLE.name)

## 3. 앱의 코드 한 줄 → 라이브러리의 어느 부품?

`stt.py`의 `load_model`이 `WhisperModel("tiny", device="cpu", compute_type="int8")`로 모델을 읽고, `recognize`가 `model.transcribe(...)`를 부릅니다. 이 한 줄 뒤에는 **세 부품**이 있습니다.

| 부품 | 정체 | 하는 일 |
|---|---|---|
| `model.feature_extractor` | NumPy로 쓴 순수 파이썬 | 파형 → Log-Mel (5절) |
| `model.model` | **CTranslate2**(C++) `Whisper` | 인코더·디코더 계산(6·7절). 층 객체는 밖에서 열 수 없음 |
| `model.hf_tokenizer` | Hugging Face `tokenizers` | 토큰 ↔ 글 (7절) |

In [ ]:
# ── 3절: 앱이 모델을 부르는 코드와 faster-whisper 안의 세 부품 ──────────────────────
# 하는 일: stt.py의 로딩 함수 소스를 보여 주고, 같은 설정으로 모델을 읽어 세 부품의 정체를 확인한다.
print(inspect.getsource(stt.load_model))
model = WhisperModel("tiny", device="cpu", compute_type="int8", download_root="models")
print("feature_extractor:", type(model.feature_extractor).__module__, type(model.feature_extractor).__name__)
print("model            :", type(model.model).__module__, type(model.model).__name__)
print("hf_tokenizer     :", type(model.hf_tokenizer).__module__, type(model.hf_tokenizer).__name__)
print("CTranslate2 모델 정보: n_mels =", model.model.n_mels, "| 다국어 =", model.model.is_multilingual, "| 언어 수 =", model.model.num_languages)
print("앱 설정(stt.settings):", stt.settings())

## 4. 모델 파일을 직접 열어 구조 세기

`model.bin`은 CTranslate2가 쓰는 **하나의 이진 파일**입니다. 구조는 단순합니다 **[확인: 아래 파서가 끝까지 읽힘]**.

```
버전(4바이트) | 모델 종류 글자("WhisperSpec") | 개정 번호 | 변수 개수
변수 × N:  이름 | 차원 수 | 각 차원 크기 | 자료형 | 바이트 수 | 값(바이트)
```

그래서 **변수 이름과 모양만 읽으면** 층이 몇 개이고 차원이 얼마인지, 파라미터가 몇 개인지를 모델을 돌리지 않고도 셀 수 있습니다. 값(가중치)은 읽지 않고 건너뜁니다.

In [ ]:
# ── 4절-1: model.bin 파서와 크기별 구조 표 ──────────────────────────────────────────
# 하는 일: model.bin의 변수 이름·모양을 읽어(값은 건너뜀) 층 수, 차원, 파라미터 수를 센다.
# [수업에 없음] 이진 파일 읽기(struct). 형식은 CTranslate2 파이썬 변환기의 쓰기 코드(model_spec.py)와 같다 [확인: 148개 변수가 끝까지 읽힘].
import struct

DTYPES = {0: "float32", 1: "int8", 2: "int16", 3: "int32", 4: "float16", 5: "bfloat16"}

def read_ct2_variables(path):
    """model.bin → [(변수 이름, 모양 리스트, 자료형, 바이트 수), ...]. 값은 읽지 않고 건너뛴다."""
    b = Path(path).read_bytes()
    pos = 0
    def u(fmt):
        nonlocal pos
        n = struct.calcsize(fmt); v = struct.unpack_from("<" + fmt, b, pos); pos += n
        return v[0]
    def text():
        nonlocal pos
        n = u("H"); t = b[pos:pos + n]; pos += n          # 길이 + 글자(끝에 \0 포함)
        return t.rstrip(b"\0").decode()
    version = u("I"); spec = text(); revision = u("I"); n_vars = u("I")
    out = []
    for _ in range(n_vars):
        name = text(); rank = u("B"); dims = [u("I") for _ in range(rank)]
        dtype = DTYPES[u("B")]; nbytes = u("I"); pos += nbytes     # 값은 건너뜀
        out.append((name, dims, dtype, nbytes))
    return out

def n_params(dims):
    return math.prod(dims) if dims else 0

rows = []
VARS = {}
for size in SIZES:
    v = read_ct2_variables(model_dir(size) / "model.bin")
    VARS[size] = v
    layers = lambda side: len({re.match(rf"{side}/layer_(\d+)/", n).group(1) for n, *_ in v if re.match(rf"{side}/layer_(\d+)/", n)})
    vocab, d_model = [d for n, d, *_ in v if n == "decoder/embeddings/weight"][0]
    enc = sum(n_params(d) for n, d, *_ in v if n.startswith("encoder/"))
    dec = sum(n_params(d) for n, d, *_ in v if n.startswith("decoder/"))
    heads = None
    rows.append({"모델": size, "인코더 층": layers("encoder"), "디코더 층": layers("decoder"), "차원 d": d_model,
                 "어휘 수": vocab, "인코더 파라미터(M)": round(enc / 1e6, 1), "디코더 파라미터(M)": round(dec / 1e6, 1),
                 "합계(M)": round((enc + dec) / 1e6, 1), "임베딩 비중(%)": round(100 * vocab * d_model / (enc + dec), 1),
                 "파일 크기(MB)": round((model_dir(size) / "model.bin").stat().st_size / 1e6)})
table = pd.DataFrame(rows).set_index("모델")
display(table)
paper = {"tiny": 39, "base": 74, "small": 244, "medium": 769}       # [일반 지식] Whisper 논문 표 1의 파라미터 수(M)
print("논문 표기(M) [일반 지식]:", {k: paper[k] for k in SIZES})
print("우리가 센 합계(M) [확인]  :", {k: table.loc[k, "합계(M)"] for k in SIZES})

**읽는 법** (위 표, 모두 **[확인]**)
- 층 수는 `tiny` 4 → `base` 6 → `small` 12 → `medium` 24로 **두 배씩**, 차원은 384 → 512 → 768 → 1024로 늘어납니다. 인코더와 디코더의 층 수는 **같습니다**.
- 우리가 센 합계는 논문의 표기(39 / 74 / 244 / 769M)와 **조금 작습니다**(약 1~3%). 논문은 반올림한 값이고, 우리는 변수에 들어 있는 값만 셌기 때문입니다 **[추정]**.
- **작은 모델일수록 임베딩(어휘 51,865개 × 차원)이 차지하는 비중이 큽니다**: `tiny`는 절반 이상이 "단어 사전"입니다. 큰 모델에서는 어텐션·FFN 층이 대부분입니다.
- 디코더가 인코더보다 큽니다(교차 어텐션과 임베딩 때문). 그런데 인코더는 **1500개 위치**를 처리해야 해서 계산량은 인코더 쪽이 큽니다 **[추정]**.

In [ ]:
# ── 4절-2: 층 하나를 열어 보기 (tiny, 층 0) ────────────────────────────────────────
# 하는 일: tiny의 인코더·디코더 층 0번 변수 이름과 모양을 출력하고, 각 변수가 수업의 어떤 개념인지 주석을 붙여 읽는다.
v = VARS["tiny"]
def show(prefix, title):
    print(f"── {title}")
    for n, d, t, b in v:
        if n.startswith(prefix) and d:
            print(f"  {n[len(prefix):]:<42} {str(d):<16} {t}")
show("encoder/conv", "인코더 합성곱(006 CNN)")
show("encoder/position_encodings/", "인코더 위치 정보")
show("encoder/layer_0/", "인코더 층 0")
show("decoder/layer_0/", "디코더 층 0")
print("LSTM·GRU 변수 개수(가설: 0):", sum(1 for n, *_ in v if re.search(r"lstm|gru|rnn", n, re.I)))

**변수를 읽어 보면** (tiny, 차원 $d=384$) **[확인: 위 출력의 모양]**

| 변수 | 모양 | 뜻 |
|---|---|---|
| `encoder/conv1` | `[384, 80, 3]` | 80개 Mel 채널 → 384 채널, 커널 길이 3 (시간 방향, 006 CNN) |
| `encoder/conv2` | `[384, 384, 3]` | **시간 길이를 3000 → 1500으로 절반**으로 줄임(stride 2는 파일에 적혀 있지 않고 [일반 지식], 6절에서 출력 모양으로 확인) |
| `…self_attention/linear_0` | `[1152, 384]` = `[3d, d]` | **Q·K·V를 한 번에** 만드는 가중치 (011 Attention) |
| `…self_attention/linear_1` | `[384, 384]` | 어텐션 결과를 섞는 출력 투영 |
| `…ffn/linear_0` → `linear_1` | `[1536, 384]` → `[384, 1536]` | 차원을 **4배로 늘렸다가 되돌리는** 피드포워드 (012 Transformer) |
| 디코더 `attention/linear_0·1·2` | `[d,d]`, `[2d,d]`, `[d,d]` | **교차 어텐션**: Q는 디코더(글)에서, K·V는 인코더 출력(소리)에서 |
| `decoder/embeddings/weight` | `[51865, 384]` | 토큰 51,865개 각각의 384차원 벡터 |
| `decoder/position_encodings` | `[448, 384]` | 디코더가 쓸 수 있는 **최대 글 길이 448토큰** |
| `encoder/position_encodings` | `[1500, 384]` | 인코더 위치 1500개(= 30초 ÷ 20 ms) |

- **LSTM·GRU 변수는 0개**입니다 → 수업 009의 RNN 계열은 Whisper에 없고 **Transformer만** 쓰입니다(가설 확인).

In [ ]:
# ── 4절-3: 파라미터가 어디에 쓰이나 (모델 크기별 구성 비율) ───────────────────────────
# 하는 일: 변수를 4묶음(임베딩 / 어텐션 / FFN / 그 밖)으로 나눠 모델 크기별 비율을 쌓은 막대로 그린다.
def group(name):
    if name == "decoder/embeddings/weight":
        return "토큰 임베딩"
    if re.match(r"(encoder|decoder)/layer_\d+/(self_attention|attention)/", name) and "layer_norm" not in name:
        return "어텐션(셀프·교차)"
    if re.match(r"(encoder|decoder)/layer_\d+/ffn/", name) and "layer_norm" not in name:
        return "FFN"
    return "그 밖(합성곱·위치·정규화)"

order = ["토큰 임베딩", "어텐션(셀프·교차)", "FFN", "그 밖(합성곱·위치·정규화)"]
share = {}
for size in SIZES:
    c = collections.Counter()
    for n, d, *_ in VARS[size]:
        if d:
            c[group(n)] += n_params(d)
    share[size] = pd.Series({g: c[g] for g in order})
share = pd.DataFrame(share)
pct = share / share.sum() * 100
fig, ax = plt.subplots(figsize=(7.5, 3.6))
left = np.zeros(len(SIZES))
colors = [theme.SERIES[0], theme.SERIES[1], theme.SERIES[3], theme.SERIES[2]]      # 초록 옆에 주황이 오지 않게 배열
for g, col in zip(order, colors):
    ax.barh(SIZES, pct.loc[g].values, left=left, color=col, label=g, edgecolor="white")
    left += pct.loc[g].values
ax.set_xlabel("파라미터 비율 (%)"); ax.set_title("Whisper 모델 크기별 파라미터 구성 [확인: model.bin]")
ax.invert_yaxis(); ax.legend(ncol=2, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, -0.45))
plt.tight_layout(); plt.show()
display(pct.round(1))

## 5. 입력: 소리는 어떤 모양으로 들어가나 (Log-Mel 80 × 3000)

수업 4강의 흐름과 **거의 같습니다**: 파형 → STFT(창으로 잘라 FFT) → 파워 → **Mel 필터** → **로그**. 다른 점은 숫자입니다.

| | 수업 4강(`visualisation.logmel`) | Whisper |
|---|---|---|
| 표본화 | 16 kHz | **16 kHz** |
| 창 길이(`n_fft`) | 수업 설정 | **400 샘플 = 25 ms** |
| 이동(`hop`) | 수업 설정 | **160 샘플 = 10 ms** → 1초에 100프레임 |
| Mel 채널 | 40 | **80** (큰 v3 모델은 128) |
| 길이 | 소리 길이만큼 | **항상 30초 = 3000프레임**(모자라면 채우고 넘으면 자름) |
| 값 정규화 | dB | $\log_{10}$ 후 **최댓값 − 8 아래는 잘라내고**, $(x+4)/4$로 정리 |

> 계획서의 "Whisper 내부 Log-Mel 80과 4강 Log-Mel 40을 비교"에서, **두 값은 설정이 달라 수치를 직접 비교하지 않고 구조(같은 파이프라인)만 비교**합니다.

In [ ]:
# ── 5절-1: 예시 소리의 파형과 Whisper 입력 Log-Mel ──────────────────────────────────
# 하는 일: 예시 TTS 소리를 16 kHz로 읽고(faster-whisper의 decode_audio), feature_extractor로 Log-Mel을 만들어 파형과 함께 그린다.
# [수업과 같음] STFT·Mel·로그 흐름(4강)  [수업에 없음] Whisper의 정규화(최댓값−8, (x+4)/4)와 30초 고정
fe = model.feature_extractor
audio16 = decode_audio(str(SAMPLE), sampling_rate=16000)
mel = fe(audio16)                                   # (80, 프레임)
print(f"소리 {len(audio16)/16000:.2f}초, {len(audio16)}샘플 → Log-Mel {mel.shape} (프레임 1개 = {fe.hop_length/fe.sampling_rate*1000:.0f} ms, 창 = {fe.n_fft/fe.sampling_rate*1000:.0f} ms)")
print(f"Mel 필터 {fe.mel_filters.shape} = (Mel 채널, FFT 칸 {fe.n_fft//2+1}), 창 하나가 만드는 FFT 칸 수 = n_fft/2+1")
mel30 = pad_or_trim(mel)                            # 30초 = 3000프레임으로 맞춤
fig, ax = plt.subplots(3, 1, figsize=(9, 7.2))
t = np.arange(len(audio16)) / 16000
ax[0].plot(t, audio16, lw=0.5, color=theme.WAVEFORM); ax[0].set_title("파형(예시 TTS)"); ax[0].set_xlabel("초")
ax[1].imshow(mel, origin="lower", aspect="auto", cmap=theme.CMAP_SPEC, extent=[0, mel.shape[1] * 0.01, 0, 80])
ax[1].set_title("Log-Mel (소리 길이만큼)"); ax[1].set_xlabel("초"); ax[1].set_ylabel("Mel 채널")
ax[2].imshow(mel30, origin="lower", aspect="auto", cmap=theme.CMAP_SPEC, extent=[0, 30, 0, 80])
ax[2].set_title("모델에 실제로 들어가는 입력: 30초(3000프레임)로 채운 Log-Mel"); ax[2].set_xlabel("초"); ax[2].set_ylabel("Mel 채널")
plt.tight_layout(); plt.show()
print("채운 부분의 값 [확인]:", float(mel30[:, mel.shape[1]:].min()), "~", float(mel30[:, mel.shape[1]:].max()),
      "| 소리 부분 값 범위:", round(float(mel.min()), 2), "~", round(float(mel.max()), 2))

In [ ]:
# ── 5절-2: 직접 만든 Log-Mel이 라이브러리와 같은가 (손계산 대조) ───────────────────────
# 하는 일: 위 설명(STFT → 파워 → Mel → log10 → 최댓값−8 → (x+4)/4)을 NumPy로 **직접** 구현해 라이브러리 결과와 비교한다.
# [수업과 같음] STFT 계산 흐름(4강)  이 셀의 목적: "설명이 코드와 같다"는 증거(검증) — 같은 쪽이 만든 기대값이 아니라 라이브러리 출력과 대조한다.
def my_logmel(x, n_fft=400, hop=160, mel_filters=None):
    x = np.pad(x.astype(np.float32), (0, 160))                    # 라이브러리가 끝에 0을 160개 붙임
    x = np.pad(x, (n_fft // 2, n_fft // 2), mode="reflect")       # 창이 가운데에 오도록 양끝을 거울처럼 채움(center=True)
    win = np.hanning(n_fft + 1)[:-1].astype(np.float32)           # 한(Hann) 창
    n_frames = 1 + (len(x) - n_fft) // hop
    frames = np.stack([x[i * hop:i * hop + n_fft] * win for i in range(n_frames)])
    power = np.abs(np.fft.rfft(frames, axis=1)) ** 2              # (프레임, 201)
    power = power[:-1].T                                          # 마지막 프레임을 버림, (201, 프레임)
    m = mel_filters @ power                                       # Mel 필터: (80, 201) @ (201, 프레임)
    log = np.log10(np.clip(m, 1e-10, None))
    log = np.maximum(log, log.max() - 8.0)                        # 가장 센 값보다 80 dB(=log 8) 아래는 잘라냄
    return (log + 4.0) / 4.0

mine = my_logmel(audio16, mel_filters=fe.mel_filters)
print("모양 같음:", mine.shape == mel.shape, "| 최대 절대 차이:", float(np.abs(mine - mel).max()))
assert np.allclose(mine, mel, atol=1e-3), "직접 만든 값과 라이브러리 값이 다릅니다"
print("→ 설명한 5단계(STFT → 파워 → Mel → log10 → 정규화)가 모델 입력과 같다 [확인]")

**확인된 것** (위 두 셀)
- 소리가 2.65초여도 모델 입력은 **항상 (80, 3000)** 입니다. 모자란 부분은 **0으로 채웁니다** — 그런데 이 0은 "무음의 Log-Mel 값"이 아닙니다(소리가 있는 부분의 값은 음수까지 내려갑니다). 즉 **채움은 모델에게 "여기는 소리가 없다"고 알려 주는 별도 신호가 아니라 그냥 0**이라서, 모델은 학습으로 이 패턴(끝에 0이 이어짐)을 "소리 끝"으로 배웠을 것입니다 **[추정]**.
- 직접 만든 Log-Mel이 라이브러리와 같습니다(최대 차이가 거의 0). 수업 4강의 STFT·Mel 지식이 **그대로** Whisper의 입력입니다.

## 6. 인코더: 소리를 "의미 있는 표현"으로

인코더는 (80, 3000) 입력을 **(1500, d)** 로 바꿉니다. 시간 방향이 절반이 되는 이유는 `conv2`의 stride가 2이기 때문입니다 → **한 위치 = 20 ms**. 그다음 **셀프 어텐션 층 N개**가 1500개 위치가 서로를 보게 합니다.

**가설**: 인코더는 **항상 30초 분량(1500 위치)** 을 처리하므로, 소리가 3초든 25초든 **계산 시간은 같을 것**입니다.

In [ ]:
# ── 6절-1: 인코더 출력의 모양과 위치별 크기 ────────────────────────────────────────
# 하는 일: 30초로 채운 Log-Mel을 인코더에 넣어 출력 모양을 보고, 위치(20 ms)마다 벡터 크기(L2)를 그려 소리 부분과 채운 부분을 비교한다.
# [수업에 없음] CTranslate2 model.encode (인코더만 따로 실행)
def encode_features(mel_chunk):
    x = np.ascontiguousarray(pad_or_trim(mel_chunk)[None]).astype(np.float32)      # (1, 80, 3000)
    return model.encode(x)

enc = encode_features(mel)
E = np.array(enc)[0]                                  # (1500, d)
print("인코더 출력 모양:", E.shape, "= (위치 1500개 = 30초/20 ms, 차원 d)")
n_speech = int(np.ceil(mel.shape[1] / 2))
norm = np.linalg.norm(E, axis=1)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(np.arange(1500) * 0.02, norm, lw=0.8, color=theme.SERIES[0])
ax.axvspan(n_speech * 0.02, 30, color="#999999", alpha=0.15, label="0으로 채운 부분")
ax.set_xlabel("초"); ax.set_ylabel("위치별 벡터 크기(L2)"); ax.set_title("인코더 출력: 위치(20 ms)마다의 표현 크기"); ax.legend()
plt.tight_layout(); plt.show()
print(f"소리 부분(앞 {n_speech}개 위치) 평균 {norm[:n_speech].mean():.1f} | 채운 부분 평균 {norm[n_speech:].mean():.1f} [확인]")

In [ ]:
# ── 6절-2: 짧은 소리와 긴 소리의 인코더 시간 ──────────────────────────────────────────
# 하는 일: 같은 소리를 이어 붙여 3초·12초·25초짜리를 만들고, 인코더 시간을 **번갈아 8번씩** 재서 중앙값·최솟값·최댓값을 본다.
# [추정 검증] 가설: 모두 (80, 3000)으로 채워지므로 시간이 같다. CPU 시간은 다른 프로세스에 따라 흔들리므로 한 번의 값은 믿지 않고 중앙값을 본다.
waves = {t_: np.tile(audio16, int(np.ceil(t_ * 16000 / len(audio16))))[: t_ * 16000] for t_ in (3, 12, 25)}
feats = {t_: fe(w) for t_, w in waves.items()}
for _ in range(2):
    encode_features(feats[3])                         # 워밍업(처음 몇 번은 느림)
times = {t_: [] for t_ in waves}
for _ in range(8):
    for t_ in waves:                                  # 번갈아 재서 컴퓨터 상태의 변화가 한쪽에만 몰리지 않게 한다
        t0 = time.perf_counter(); encode_features(feats[t_]); times[t_].append(time.perf_counter() - t0)
display(pd.DataFrame([{"소리 길이(초)": t_, "Log-Mel 프레임": feats[t_].shape[1], "모델 입력": "(80, 3000)",
                       "중앙값(초)": round(float(np.median(v_)), 2), "최소(초)": round(min(v_), 2), "최대(초)": round(max(v_), 2)}
                      for t_, v_ in times.items()]))
print("측정 시점 컴퓨터 부하(load average 1·5·15분):", tuple(round(x, 1) for x in os.getloadavg()))

**읽는 법 [확인: 이 컴퓨터(CPU 4스레드, `tiny`, `int8`)]**
- 입력 모양이 모두 (80, 3000)이라서 **중앙값이 서로 비슷합니다**(가설 확인). 같은 입력도 측정마다 **몇 초씩 흔들리므로**(최소·최대 열) 한 번의 값이 아니라 중앙값을 봅니다 — 첫 시도에서는 3초가 5초, 12초가 2.5초로 나와 "짧을수록 빠르다/느리다"로 읽을 뻔했고, 번갈아 8번 재서 흔들림임을 확인했습니다 **[확인]**.
- 소리가 3초든 25초든 **비용은 "소리 길이"가 아니라 "30초 창이 몇 개냐"** 로 정해집니다.
- 그래서 우리 앱의 짧은 한 문장(2~3초) 인식도 **30초 창 한 개 분량의 인코더 시간**이 듭니다. 처음 호출이 유난히 느린 것은 모델 로딩과 워밍업 때문입니다.
- 노래 구간(10~20초)도 같은 30초 창 하나라서, **신뢰 지표(`avg_logprob` 등)가 줄별이 아니라 창 하나의 값**이 됩니다(그래서 앱은 이 지표를 클립 단위 힌트로만 씁니다, `stt.recognize_segments`).

## 7. 디코더: 글을 한 토큰씩 쓰기 (자기회귀)

디코더는 지금까지 쓴 토큰 열과 인코더 출력을 보고 **다음 토큰의 확률분포**를 냅니다. 그중 하나를 고르고(우리 앱은 `beam_size=1` = 가장 높은 것만, 탐욕 디코딩), 그 토큰을 입력 끝에 붙여 **다시** 다음을 구합니다. `<|endoftext|>`가 나오면 끝입니다.

In [ ]:
# ── 7절-1: 토큰 — 특수 토큰, 글 조각, 언어별 토큰 수 ──────────────────────────────────
# 하는 일: 디코더가 쓰는 특수 토큰의 번호를 보여 주고, 문장이 몇 개의 토큰이 되는지(언어마다 다른지) 센다.
# [수업에 없음] 토크나이저(byte-level BPE): 글을 "자주 나오는 글자 조각"으로 나눈 번호 열
tok = Tokenizer(model.hf_tokenizer, model.model.is_multilingual, task="transcribe", language="en")
hf = model.hf_tokenizer
specials = {"<|startoftranscript|>": tok.sot, "<|en|>": hf.token_to_id("<|en|>"), "<|ja|>": hf.token_to_id("<|ja|>"),
            "<|transcribe|>": tok.transcribe, "<|notimestamps|>": tok.no_timestamps, "<|endoftext|>": tok.eot,
            "첫 타임스탬프 <|0.00|>": tok.timestamp_begin}
display(pd.DataFrame({"토큰 번호": specials}))
n_text = hf.get_vocab_size()
emb_rows = [d for n, d, *_ in VARS["tiny"] if n == "decoder/embeddings/weight"][0][0]
print(f"글·특수 토큰 {n_text:,}개 + 타임스탬프 토큰 {emb_rows - n_text:,}개 = 모델의 임베딩 행 {emb_rows:,}개 [확인: 4절 변수 모양과 계산이 맞음]")
print("타임스탬프 토큰: 번호가 1 늘 때마다 0.02초 → 0.00 ~ 30.00초 = 1,501개 [일반 지식]")
samples = {"영어": "I want to explore the universe.", "일본어": "朝の光が部屋を満たしていく", "한국어": "아침 햇살이 방을 채워요"}
rows = []
for lang, s in samples.items():
    ids = hf.encode(s, add_special_tokens=False)
    rows.append({"언어": lang, "문장": s, "글자 수": len(s), "토큰 수": len(ids.ids), "토큰 조각(앞 6개)": ids.tokens[:6]})
display(pd.DataFrame(rows))
print("※ 'Ġ'는 앞의 공백, 일본어·한국어 조각이 'æľĿ' 같은 이상한 글자로 보이는 것은 UTF-8 **바이트**를 보이는 글자로 바꿔 적은 표기다(byte-level BPE) [일반 지식].")

In [ ]:
# ── 7절-2: 언어 감지 — 디코더의 첫 토큰 하나 ───────────────────────────────────────────
# 하는 일: 인코더 출력만 보고 "이 소리는 어느 언어인가"를 구한다(디코더가 <|startoftranscript|> 다음 토큰으로 언어 토큰을 고르는 확률).
# 앱에서는 stt.detect_language가 이 기능으로 '내 노래'의 언어를 제안한다(사용자가 확인).
lang_probs = model.model.detect_language(enc)[0]
display(pd.DataFrame(lang_probs[:5], columns=["언어 토큰", "확률"]).round(4))

In [ ]:
# ── 7절-3: 토큰을 하나씩 — 단계마다 상위 5개 후보와 확률 ─────────────────────────────
# 하는 일: 예시 소리를 디코더로 풀면서, 각 단계에서 모델이 **어떤 후보를 얼마의 확률로** 보았는지(상위 5개)를 그린다.
# 모델이 내는 logits(점수)에 softmax를 씌우면 확률이 된다.
prompt = list(tok.sot_sequence) + [tok.no_timestamps]          # <|startoftranscript|><|en|><|transcribe|><|notimestamps|>
res = model.model.generate(enc, [prompt], beam_size=1, max_length=60, return_scores=True, return_logits_vocab=True, return_no_speech_prob=True)[0]
ids = res.sequences_ids[0]
logits = np.array(res.logits[0])                       # (단계, 어휘)  — 마지막 단계는 <|endoftext|>를 고르는 단계
def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()
print("프롬프트:", [hf.id_to_token(i) for i in prompt])
print("생성된 토큰:", [hf.id_to_token(i) for i in ids], "→", tok.decode(ids))
print(f"평균 로그확률 avg_logprob = {res.scores[0]:.3f} | 소리 없음 확률 no_speech_prob = {res.no_speech_prob:.3f}")
n_steps = logits.shape[0]
fig, axes = plt.subplots(2, (n_steps + 1) // 2, figsize=(11, 4.2)); axes = axes.ravel()
for t in range(n_steps):
    p = softmax(logits[t]); top = np.argsort(p)[::-1][:5]
    labels = [hf.id_to_token(int(i)) or "?" for i in top]
    chosen = ids[t] if t < len(ids) else tok.eot
    cols = [theme.SERIES[0] if int(i) == chosen else "#9bb3a3" for i in top]
    axes[t].barh(labels[::-1], p[top][::-1], color=cols[::-1]); axes[t].set_xlim(0, 1)
    axes[t].set_title(f"단계 {t+1}", fontsize=9); axes[t].tick_params(labelsize=7)
for k in range(n_steps, len(axes)):
    axes[k].axis("off")
fig.suptitle("단계마다 상위 5개 후보(초록 = 실제로 고른 토큰)", y=1.0); plt.tight_layout(); plt.show()

In [ ]:
# ── 7절-4: 타임스탬프 토큰 — 시간도 "글자"로 쓴다 ─────────────────────────────────────
# 하는 일: <|notimestamps|>를 빼고 풀면 디코더가 <|0.00|> 같은 시간 토큰을 글 사이에 끼워 쓴다. 번호 차가 0.02초임을 확인한다.
# 우리 앱의 줄 시간(stt.recognize_segments)과 단어별 시간은 이 토큰과, 단어 시간은 교차 어텐션 정렬(alignment_heads)로 얻는다.
prompt_ts = list(tok.sot_sequence)                    # <|notimestamps|> 없이
r2 = model.model.generate(enc, [prompt_ts], beam_size=1, max_length=60)[0]
ids_ts = r2.sequences_ids[0]
print("토큰 번호:", ids_ts)
print("시간 토큰으로 읽기:", tok.decode_with_timestamps(ids_ts))
tsb = tok.timestamp_begin
print("시간 토큰 번호 → 초:", [(i, round((i - tsb) * 0.02, 2)) for i in ids_ts if i >= tsb], "[확인: (번호 − 첫 시간 토큰) × 0.02]")

**확인된 것** (위 셀들)
- 디코더는 **확률분포에서 하나를 고르는 일**을 반복합니다. 소리가 또렷한 예시에서는 첫 후보가 압도적입니다. 후보 표를 보면 "그럴듯한 다음 말"의 후보들(`Want`, `wanted` …)이 함께 보입니다 → **디코더가 언어 모델이라는 증거**입니다.
- 같은 뜻의 문장도 **언어마다 토큰 수가 다릅니다**(위 표: 영어는 약 3.4글자당 토큰 1개인데 일본어·한국어는 글자 1개당 거의 토큰 1개 **[확인: 예문 1개씩]**). 긴 일본어 가사는 같은 길이의 영어 가사보다 더 많은 단계가 필요하고 틀릴 기회도 늘어납니다 **[추정]**. (우리 앱이 일본어를 단어(WER)가 아닌 글자(CER)로 채점하는 것과는 별개의 이야기입니다.)
- **시간도 토큰**입니다. 번호가 1 늘면 0.02초이고, 이 분해능이 줄 시간의 한계(20 ms)입니다.

## 8. 우리가 겪은 현상을 구조로 설명하기

| 우리가 겪은 일 | 구조에서 나오는 이유 | 이 노트북의 증거 |
|---|---|---|
| **무음·잡음 입력에서 말을 만들어 냄**(환각) | 디코더는 언어 모델이라 입력이 약해도 "그럴듯한 말"을 고른다. 확률은 낮아도 **고른다** | 8절-1 (잡음), 8절-2 (무음) |
| **`no_speech_prob`·`avg_logprob`로 걸러 냄** | 모델이 `<|nospeech|>` 확률과 고른 토큰들의 평균 로그확률을 함께 낸다. 둘 다 나쁘면 그 창을 버림(faster-whisper 기본 규칙) | 8절-2 |
| **반복 루프**(같은 말을 계속 씀, `temperature=0.0`일 때) | 이미 쓴 토큰이 다음 토큰의 확률을 높여 **자기 강화**된다(자기회귀의 약점). 압축률이 비정상적으로 높아진다 | 8절-3 |
| **온도 폴백**(기본 설정) | 압축률 > 2.4 또는 평균 로그확률 < −1.0이면 **같은 창을 온도를 올려 다시** 디코딩 | 8절-3, 8절-4 |
| **원곡에 VAD를 켜면 가사가 통째로 사라짐** | VAD(Silero)는 "말소리 vs 나머지"를 가르는 별도 작은 모델인데, **노래의 반주 위 보컬을 "말소리가 아님"으로 판정**해 창을 아예 모델에 넣지 않는다 [검토 문서 13.5, 원곡 대상 실험] | 이 노트북에서는 상업곡을 쓰지 않아 **재현하지 않음 [미확인: 재현]** |
| **모델이 클수록 틀린 발음을 더 잘 "고쳐 적음"** | 큰 디코더는 언어 지식이 더 많아 소리가 불분명해도 문맥으로 정답에 가깝게 쓴다 → 그래서 점수는 **발음 점수가 아니라 문장 인식 일치도** | 4절(디코더가 더 큼), 7절 후보 표 |
| **신뢰 지표가 줄이 아니라 창 하나의 값** | 인코더·디코더가 30초 창 단위로 돈다 | 6절-2 |

In [ ]:
# ── 8절-1: 잡음을 키우면 확률은 어떻게 변하나 (시드 3개) ──────────────────────────────
# 하는 일: 예시 소리에 흰 잡음을 SNR 20/10/0/-5 dB로 섞어(시드 3개씩) tiny가 내는 글, 평균 로그확률, 소리 없음 확률을 본다.
# 합성 흰 잡음 결과를 실제 환경으로 일반화하지 않는다(프로젝트 원칙). 우리 앱의 설정과 같이 beam_size=1.
def decode_wave(wave):
    e = encode_features(fe(wave))
    r = model.model.generate(e, [prompt], beam_size=1, max_length=100, return_scores=True, return_no_speech_prob=True)[0]
    return tok.decode(r.sequences_ids[0]).strip(), r.scores[0], r.no_speech_prob

def add_noise(wave, snr_db, seed):
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(len(wave)).astype("float32") * np.sqrt(np.mean(wave ** 2) / 10 ** (snr_db / 10))
    return wave + noise

_t, _lp, _ns = decode_wave(audio16)
rows = [{"SNR(dB)": "깨끗함", "시드": "-", "인식 글": _t, "avg_logprob": round(_lp, 2), "no_speech_prob": round(_ns, 2)}]
for snr in (20, 10, 0, -5):
    for seed in (1, 2, 3):
        text_, lp, ns = decode_wave(add_noise(audio16, snr, seed))
        rows.append({"SNR(dB)": snr, "시드": seed, "인식 글": text_, "avg_logprob": round(lp, 2), "no_speech_prob": round(ns, 2)})
noise_table = pd.DataFrame(rows)
display(noise_table)
print("faster-whisper 기본 임계값: avg_logprob < -1.0 이면 불안정 판정, no_speech_prob > 0.6 이면 소리 없음 후보")

In [ ]:
# ── 8절-2: 무음에서 말을 만드나? (환각) ────────────────────────────────────────────────
# 하는 일: 5초 무음과 아주 작은 잡음을 디코더에 넣어, 무엇을 쓰는지·확률이 어떤지 보고 faster-whisper의 건너뛰기 규칙에 적용해 본다.
# 규칙(faster-whisper): no_speech_prob > no_speech_threshold(0.6) 이면서 avg_logprob < log_prob_threshold(-1.0)이면 그 창을 소리 없음으로 보고 건너뜀.
rng = np.random.default_rng(0)
cases = {"5초 무음": np.zeros(16000 * 5, dtype="float32"),
         "아주 작은 잡음": (rng.standard_normal(16000 * 5) * 0.003).astype("float32"),
         "예시 소리(깨끗함)": audio16}
rows = []
for name, w in cases.items():
    text_, lp, ns = decode_wave(w)
    skipped = ns > 0.6 and lp < -1.0
    rows.append({"입력": name, "디코더가 쓴 글": repr(text_), "avg_logprob": round(lp, 2), "no_speech_prob": round(ns, 2), "faster-whisper가 건너뜀?": "예" if skipped else "아니오"})
display(pd.DataFrame(rows))

In [ ]:
# ── 8절-3: 반복 루프와 압축률 ──────────────────────────────────────────────────────────
# 하는 일: 정상 문장과 같은 말이 반복되는 글의 gzip 압축률(faster-whisper가 쓰는 지표)을 비교한다.
# 반복이 많으면 압축이 잘 되어(원본/압축 비율↑) 압축률이 커진다 → 임계값 2.4를 넘으면 온도 폴백을 한다.
import zlib
def compression_ratio(text_):
    b = text_.encode("utf-8"); return len(b) / len(zlib.compress(b))
normal = "I want to explore the universe, and then we sing together until the morning comes."
looped = "I want to explore the universe." + " the" * 30
for name, s in {"정상 문장": normal, "반복 루프": looped}.items():
    print(f"{name:<8} 압축률 {compression_ratio(s):.2f}  {'→ 임계값 2.4 초과: 온도 폴백 대상' if compression_ratio(s) > 2.4 else ''}")

In [ ]:
# ── 8절-4: faster-whisper의 기본 설정과 우리 앱의 설정 ─────────────────────────────────
# 하는 일: transcribe()의 기본값(온도 폴백·임계값 등)을 라이브러리에서 읽어 우리 앱의 설정과 나란히 놓는다.
sig = inspect.signature(WhisperModel.transcribe).parameters
wanted = ["beam_size", "temperature", "compression_ratio_threshold", "log_prob_threshold", "no_speech_threshold",
          "condition_on_previous_text", "vad_filter", "word_timestamps", "initial_prompt"]
ours = {"beam_size": "1 (stt.BEAM_SIZE)", "vad_filter": "말소리 연습: stt.recognize 끔 / words 켬 · 노래: 끔",
        "temperature": "기본(폴백 켬), 노래에서 0.0은 반복 루프를 낳아 쓰지 않음(검토 문서 13.5)", "word_timestamps": "줄 나누기·틀린 구간에서 켬"}
display(pd.DataFrame([{"설정": k, "faster-whisper 기본값": sig[k].default, "우리 앱": ours.get(k, "기본값")} for k in wanted]))

**종합 [확인 + 추정]**
- 숫자로 본 신뢰 지표: 깨끗할 때 `avg_logprob`는 0에 가깝고(−0.06), 잡음이 0 dB까지는 글이 그대로이다가 **−5 dB에서 글이 바뀌고 `avg_logprob`가 −1.0 근처로 내려갑니다**. 무음에서는 디코더가 **무언가를 씁니다**(환각) — 그러나 `no_speech_prob`가 높고 `avg_logprob`가 매우 낮아 **faster-whisper의 규칙이 걸러 냅니다**.
- 그래서 우리가 겪은 "무음·반주 구간에서 말이 생김"은 **디코더가 언어 모델이라서 생기는 일이고, 확률 지표는 그것을 알아챌 단서**입니다. 하지만 이 지표도 **창 하나당 하나**라서 노래의 줄별 신뢰도로는 못 씁니다.
- 환각을 줄이는 방법은 구조에서 바로 나옵니다: ① 소리가 없는 창을 모델에 **안 넣기**(VAD, 단 노래에서는 역효과) ② 확률이 낮으면 **다시 디코딩**(온도 폴백) ③ 더 큰 모델로 **소리 증거를 더 잘 쓰기**.

## 9. 확인 질문과 답 (면담·발표 대비)

- **Whisper는 CNN인가 RNN인가 Transformer인가?** 입력 앞에 합성곱 2개(CNN)가 있고, 본체는 **Transformer 인코더–디코더**입니다. LSTM·GRU 변수는 0개 [확인].
- **왜 항상 30초인가?** 인코더가 위치 1500개(= 30초 ÷ 20 ms)로 **고정 설계**되어 있기 때문입니다. 짧으면 0으로 채우고 길면 자르며, 긴 소리는 30초씩 잘라 반복합니다 [확인: 위치 정보 변수 `[1500, d]`, 입력 `(80, 3000)`].
- **모델 크기가 커지면 무엇이 커지나?** 층 수(4 → 24)와 차원(384 → 1024). 작은 모델은 파라미터의 절반이 **어휘 임베딩**이고, 큰 모델은 어텐션·FFN 층이 대부분 [확인: 4절].
- **디코더는 소리를 어떻게 "듣나"?** 디코더 각 층의 **교차 어텐션**이 인코더 출력(1500 위치)을 참조합니다 [확인: `attention/linear_1`이 `[2d, d]` = K·V]. 글은 한 토큰씩 나오고 그때마다 다시 계산됩니다.
- **왜 점수가 발음 점수가 아닌가?** 디코더가 언어 모델이라 틀린 발음도 문맥으로 정답에 가깝게 쓰기 때문입니다. 그래서 우리 앱은 **문장 인식 일치도**라고 말하고, 발음 평가는 별도(Azure)로 보여 줍니다.
- **tiny가 아니라 medium을 노래에 쓰는 이유는?** 노래는 소리가 어렵고 반주가 섞여 디코더의 언어 지식이 더 필요합니다. 11-03 비교에서 `medium`이 원곡에서 가장 좋았습니다(micro WER 0.177 대 0.257, 검토 문서 13.5·15.2). 이 노트북에서는 재현하지 않았습니다 **[미확인: 재현]**.

## 10. 발표에 가져갈 후보와 한계

> 이 노트북은 발표 자료가 아니라 **내가 Whisper의 구조를 이해하고 직접 확인하는 노트북**이므로 위 1~9절의 내용은 **하나도 줄이지 않고 전부 유지**합니다. 아래는 나중에 발표 자료를 만들 때 **여기서 가져다 쓸 수 있는 후보 목록**일 뿐 지금 해야 할 일이 아닙니다. 몇 개를 쓸지는 발표 시간이 정해진 뒤에 정합니다(스켈레톤 대본 초안은 3분 기준이고 모델 장을 넣으면 3분 15초로 넘음).

**발표에 가져갈 수 있는 그림·숫자 (후보)**
- **그림 ①** 4절-3: 모델 크기별 파라미터 구성 — "작은 모델의 절반은 단어 사전" (한 장으로 구조 설명)
- **그림 ②** 5절-1: 30초로 채운 Log-Mel — "수업 4강의 Mel이 그대로 입력" + 짧은 소리도 30초 창
- **표 ③** 8절-1·2: 잡음 SNR별 `avg_logprob`와 무음 환각 — "우리가 겪은 환각이 구조에서 나온다"
- (예비) 7절-3 단계별 후보 그림 — "디코더는 언어 모델"

**한계 (그대로 말하기)**
- 입력은 내 직접 쓴 예문의 TTS와 합성 흰 잡음입니다. **실제 노래·다른 화자·실제 환경으로 일반화하지 않습니다.**
- 시간 측정은 이 컴퓨터(CPU 4스레드, `tiny`, `int8`) 한 대의 값입니다.
- 층 안쪽의 **가중치 값·어텐션 패턴**은 열어 보지 않았습니다(CTranslate2 변환 모델이라 층 객체 접근이 안 됨). 어텐션 시각화는 PyTorch판 Whisper가 필요하며 별도 환경이 필요합니다 **[미확인]**.
- VAD가 노래 가사를 지우는 현상의 **재현**은 상업곡이 필요해 이 노트북에 넣지 않았습니다(결과는 검토 문서 13.5에 숫자로만).

**더 해 볼 것 (선택)**: 같은 소리를 `tiny`/`small`/`medium`에 넣어 단계별 후보 확률을 비교, 교차 어텐션 정렬(`alignment_heads`)로 단어 시간이 나오는 과정 따라가기, 논문(*Robust Speech Recognition via Large-Scale Weak Supervision*, Radford 외 2022) 읽기.

## 11. 참고와 소스 위치

- 논문: Radford 외 (2022), *Robust Speech Recognition via Large-Scale Weak Supervision* [일반 지식]
- `stt.py`: `load_model`, `recognize`, `recognize_segments`, `detect_language` / 검토 문서 13.5(노래 STT 비교), 15.2(모델 크기 비교)
- faster-whisper 소스: `faster_whisper/transcribe.py`(`transcribe`, `generate_with_fallback`), `feature_extractor.py`(Log-Mel)
- CTranslate2: `Whisper.encode`, `Whisper.generate`, `Whisper.detect_language`
- 수업: 7강 `lesson07_whisper_minimal`, 4강 STFT·Mel(`visualisation.logmel`), 011 Attention, 012 Transformer